# NCBI API request to retrieve COG data

## Introduction

This code makes API requests to the NCBI COG database to retrieve gene COG data for a specific assembly. The retrieved JSON data is parsed into a Python dictionary, and the gene name and COG are printed to an output file.


##  Setup

In [89]:
# Libraries
import requests
import pandas as pd
import time
from requests.exceptions import HTTPError, Timeout, RequestException


In [90]:
# Flags/config

# genome assembly for analysis
assembly_id = "GCF_000018685.1"


In [91]:
# File IO

# output file
output_file = f"/home/tolonen/Drives/Genoscope-googledrive/Lab_Projects/Genome_Cphy/COG/cogs_{assembly_id}.txt"

## Functions

In [92]:
# function to make GET request and return dictionary of results

def main_request(url):
    """
    Goal: 
        Fetches gene COG data from the specified NCBI API URL. 
        Parses JSON into dictionary, and returns dictionary.
    
    Args:
        url (str): The NCBI COG API URL for a specific assembly.
    """

    print(f"Attempting to fetch data from: {url}")

    try:   
        # GET request to the API
        response = requests.get(url, timeout=10)
        
        # Check if the request was successful
        response.raise_for_status()
        
        # Parse the JSON response
        data = response.json()
    except Timeout:
        print(f"Request timed out for URL: {url}")
        raise
    except RequestException as e:
        print(f"Request error: {e}")
        raise
    else:
        print("Data fetched successfully.")
        return data
    

In [93]:
# function to parse JSON data into dictionary: gene_tag -> COG ID

def get_dictionary(data, all_cogs):
    for item in data['results']:
        gene_tag = item['gene_tag']
        cog_id = item['cog']['cogid']
        all_cogs[gene_tag] = cog_id
    return all_cogs

## Main


In [ ]:

# declare all_cogs dictionary
all_cogs = {}

# initialize page number 
page = 1

# loop through API pages, collect data into dictionary: gene_tag -> COG ID
while True:
    url = f"https://www.ncbi.nlm.nih.gov/research/cog/api/cog/?assembly={assembly_id}&format=json&page={page}"

    try:
        # get API data as dictionary
        data = main_request(url)

        # Safety check: prevents 'NoneType' error if main_request fails to raise
        if data is None:
            print(f"Request failed unexpectedly and returned no data for page {page}. Stopping.")
            break

    # --- Error Handling ---
    except HTTPError as err:
        print(f"HTTP Error occurred (Status {err.response.status_code}): {err}")
        break
    except (Timeout, RequestException) as e:
        print(f"A connection or request error occurred: {e}")
        break
    except Exception as e:
        # Catch any other unexpected error (e.g., issues with JSON parsing)
        print(f"An unexpected error occurred: {e}")
        break

    else:
        # make dictionary of COG data
        all_cogs = get_dictionary(data, all_cogs)

        # Prepare for Next Iteration
        page += 1
        print(f"extracted COG data from page {page}")
        time.sleep(1) # Be kind to the API server! Add a delay.


print(f"\nSuccessfully retrieved {len(all_cogs)} gene records for assembly GCF_000018685.1.")

In [ ]:
# print COG dictionary to output file

with open(output_file, 'w') as my_out:
    print(f"COGs for each gene in genome {assembly_id} were extracted from the NCBI API using api-request_COG.ipynb\n", file = my_out)
    for key, value in all_cogs.items():
        print(f"Gene: {key}\tCOG: {value}", file = my_out)